In [9]:

import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display, clear_output
import numpy as np
%matplotlib nbagg
%matplotlib inline
import seaborn as sns
sns.set_style("whitegrid")
sns.set_palette(sns.dark_palette("purple"))

try:
    from plotting import plot_autoencoder_stats
except Exception as ex:
    print(f"If using Colab, you may need to upload `plotting.py`. \
          \nIn the left pannel, click `Files > upload to session storage` and select the file `plotting.py` from your computer \
          \n---------------------------------------------")
    print(ex)


In [10]:
import torch
cuda = torch.cuda.is_available()
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torchvision import transforms

# Flatten the 2d-array image into a vector
#flatten = lambda x: ToTensor()(x).view(28**2) # for linear ffn
transform_cnn = transforms.ToTensor()

# Define the train and test sets
dset_train = MNIST("./", train=True,  transform=transform_cnn, download=True)
dset_test  = MNIST("./", train=False, transform=transform_cnn)

# The digit classes to use
classes = [1, 2, 3, 7, 9]

def stratified_sampler(labels, classes):
    """Sampler that only picks datapoints corresponding to the specified classes"""
    from functools import reduce
    (indices,) = np.where(reduce(lambda x, y: x | y, [labels.numpy() == i for i in classes]))
    indices = torch.from_numpy(indices)
    return SubsetRandomSampler(indices)


# The loaders perform the actual work
batch_size = 64
train_loader = DataLoader(dset_train, batch_size=batch_size,
                          sampler=stratified_sampler(dset_train.targets, classes), pin_memory=cuda)
test_loader  = DataLoader(dset_test, batch_size=batch_size, 
                          sampler=stratified_sampler(dset_test.targets, classes), pin_memory=cuda)

In [ ]:
import torch.nn as nn
def double_conv2d(in_channels, out_channels, kernel, stride=1, padding = 0):
    nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, stride, padding),
        nn.BatchNorm2d(),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=kernel, stride=stride, padding=padding),
        nn.BatchNorm2d(),
        nn.ReLU()
        )
    
class encoderblock(nn.Module):
    def __init__(self, in_channels, out_channels):

        self.c = double_conv2d(in_channels, out_channels, 3)
        self.p = nn.MaxPool2d(2)

    def forward(self, x):
        
        x1 = self.c(x)
        x2 = self.p(x1)

        return x1, x2
    

class decoderblock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_chan):

        self.upc = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.c   = double_conv2d(out_channels + skip_chan, out_channels)

        

IndentationError: expected an indented block after function definition on line 2 (3938354400.py, line 3)